# Exercicio

## Parte 1: Configuração e Criação do Banco de Dados


In [1]:
import sqlite3

In [2]:
conn = sqlite3.connect("empresa.db")
cursor = conn.cursor()

#### Criando tabela funcionarios:

Crie uma tabela chamada funcionários com as seguintes colunas:

id(INTEGER, chave primária, incremento automático)

nome(TEXTO, não nulo)

cargo(TEXTO, não nulo)

salario(REAL)

data_contratacao(TEXTO, não nulo)

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS funcionarios(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    cargo TEXT NOT NULL,
    salario REAL,
    data_contratacao TEXT NOT NULL
);
""")

#### Verificação da tabela

In [4]:
cursor.execute('''PRAGMA table_info(funcionarios);''')
for row in cursor.fetchall():
    print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'nome', 'TEXT', 1, None, 0)
(2, 'cargo', 'TEXT', 1, None, 0)
(3, 'salario', 'REAL', 0, None, 0)
(4, 'data_contratacao', 'TEXT', 1, None, 0)


## Parte 2: Inserção de dados (DML)
#### Insert 

In [5]:
cursor.execute("""
    INSERT INTO funcionarios(nome, cargo, salario, data_contratacao)
    VALUES
        ('João da Silva', 'Zelador', 1600.00, '01/02/2026'),
        ('Leonardo da Vinci', 'Gerente', 5600.00, '19/04/2008'),
        ('Leticia Almeida', 'Diretor', 8000.00, '21/08/2000'),
        ('Marcos da Praia', 'Cabo', 4000.00, '01/02/2026'),
        ('Mohammad', 'CEO', 12000.00, '11/09/2001');
""")

#### Select

In [8]:
cursor.execute("""
    SELECT * FROM funcionarios
    WHERE salario > 5000 AND cargo = "Gerente"
""")

cursor.fetchall()

[(2, 'Leonardo da Vinci', 'Gerente', 5600.0, '19/04/2008')]

#### Update

In [9]:
cursor.execute('UPDATE funcionarios SET salario = (salario + (salario*0.1)) WHERE nome = "Marcos da Praia"')
conn.commit()
print("Salario de Marcos foi atualizado!")

Salario de Marcos foi atualizado!


#### Delete

In [10]:
cursor.execute("DELETE FROM funcionarios WHERE nome = 'João da Silva'")
conn.commit()


In [11]:
cursor.execute("SELECT COUNT(*) FROM funcionarios")
cursor.fetchone()

(4,)

## Parte 3: Agregação e Ordenação

#### Funções de agregação ( SUM, AVG, MAX, MIN):

In [16]:
cursor.execute("""
    SELECT SUM(salario) AS total_salario,
            AVG(salario) AS salario_medio,
            MAX(salario) AS maior_salario,
            MIN(salario) AS menor_salario
    FROM funcionarios;
""")
cursor.fetchall()

[(30000.0, 7500.0, 12000.0, 4400.0)]

#### Agrupamento de dados (GROUP BY):

In [17]:
cursor.execute("""
    SELECT cargo,
            AVG(salario) AS salario_medio_cargo,
            COUNT(id)
    FROM funcionarios
    GROUP BY cargo;
""")
cursor.fetchall()

[('CEO', 12000.0, 1),
 ('Cabo', 4400.0, 1),
 ('Diretor', 8000.0, 1),
 ('Gerente', 5600.0, 1)]

#### Ordenação (ORDER BY):

In [18]:
cursor.execute("""
    SELECT *
    FROM funcionarios
    ORDER BY salario DESC;
""")
cursor.fetchall()

[(5, 'Mohammad', 'CEO', 12000.0, '11/09/2001'),
 (3, 'Leticia Almeida', 'Diretor', 8000.0, '21/08/2000'),
 (2, 'Leonardo da Vinci', 'Gerente', 5600.0, '19/04/2008'),
 (4, 'Marcos da Praia', 'Cabo', 4400.0, '01/02/2026')]

In [21]:
cursor.execute("""
    SELECT *
    FROM funcionarios
    ORDER BY nome ASC;
""")
cursor.fetchall()

[(2, 'Leonardo da Vinci', 'Gerente', 5600.0, '19/04/2008'),
 (3, 'Leticia Almeida', 'Diretor', 8000.0, '21/08/2000'),
 (4, 'Marcos da Praia', 'Cabo', 4400.0, '01/02/2026'),
 (5, 'Mohammad', 'CEO', 12000.0, '11/09/2001')]

## Parte 4: Transações e ROLLBACK

#### Simulação de transação:

In [ ]:
try:
    conn.execute("BEGIN;")
    cursor.execute("""
            INSERT INTO funcionarios (nome, cargo, salario, data_contratacao)
            VALUES ("Vitor de Freitas", "", 12000, "12/06/2017");
        """) # Criando novo registro simulando erro (sem cargo)
    conn.commit() # Se chegasse até aqui, faria commit
except:
    conn.rollback() # Reverte as alterações em caso de erro
    print("Transação revertida por erro")

Transação revertida por erro


#### Transação com COMMIT:

In [ ]:
try:
    conn.execute("BEGIN;")
    cursor.execute('''
        INSERT INTO funcionarios (nome, cargo, salario, data_contratacao) 
        VALUES ("Vitor de Freitas", "CEO", 12000, "12/06/2017");
        ''') # Criando novo registro
    conn.commit() # Se chegasse até aqui, faria commit
except:
    conn.rollback() # Reverte as alterações em caso de erro
    print("Transação revertida por erro")

## Parte 5: Joins e Múltiplas Tabelas


#### Criação de uma segunda tabela (departamentos):

In [31]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS departamentos(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_departamento TEXT NOT NULL,
    localizacao TEXT
    );
""")

#### Adicionando valores

In [32]:
cursor.execute("""
    INSERT INTO departamentos(nome_departamento, localizacao)
    VALUES
        ('RH', 'Bairro do Limoeiro'),
        ('Marketing', 'Bairro Tatuapé'),
        ('Negócios', 'Bairro da Lapa');
""")

#### Adicionando coluna nova na tabela funcionarios:

In [ ]:
cursor.execute('''ALTER TABLE funcionarios ADD COLUMN departamento_id INTEGER REFERENCES departamentos(id);''')
conn.commit()
print("Coluna departamento_id adicionada com sucesso!")

Coluna categoria adicionada com sucesso!


#### Adicionando valores na nova coluna departamento_id:

In [ ]:
# Adicionando o departamento_id na tabela funcionários
dados_atualizacao = [
    (2, "Leonardo da Vinci"),
    (3, "Leticia Almeida"),
    (1, "Marcos da Praia"),
    (2, "Mohammad"),
    (1, "Vitor de Freitas")
]

cursor.executemany('''
    UPDATE funcionarios 
    SET departamento_id = ? 
    WHERE nome = ?;
''', dados_atualizacao)

conn.commit()
print(f"Sucesso! {len(dados_atualizacao)} funcionários foram atualizados.")


Sucesso! 5 funcionários foram atualizados.


#### Inner join

In [36]:
cursor.execute('''
    SELECT f.nome, d.nome_departamento
    FROM funcionarios f
    INNER JOIN departamentos d ON d.id = f.departamento_id
''')

cursor.fetchall()

[('Leonardo da Vinci', 'Marketing'),
 ('Leticia Almeida', 'Negócios'),
 ('Marcos da Praia', 'RH'),
 ('Mohammad', 'Marketing'),
 ('Vitor de Freitas', 'RH')]

### Criando tabelas para left join

#### Criando tabela projetos

In [37]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS projetos(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_projeto TEXT NOT NULL
    );
""")

#### Criando tabela auxiliar funcionarios_projetos com o id do funcionario e o id do projeto

In [38]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS funcionarios_projetos(
    funcionario_id INTEGER REFERENCES funcionarios(id),
    projeto_id INTEGER REFERENCES projetos(id),
    PRIMARY KEY (funcionario_id, projeto_id)
);
""")


#### Adicionando projetos

In [39]:
cursor.execute("""
    INSERT INTO projetos(nome_projeto)
    VALUES
        ('Dominos'),
        ('Madero'),
        ('BurguerKing');
""")

#### Adicionando valores na tabela funcionarios_projetos

In [ ]:
vinculos_dados = [
    (1, 1), 
    (2, 1),  
    (2, 2),  
    (3, 3),  
    (4, 2)  
]
cursor.executemany("INSERT INTO funcionarios_projetos (funcionario_id, projeto_id) VALUES (?, ?);", vinculos_dados)
conn.commit()


#### Left Join
Mostra o funcionário, o departamento que ele é, e por fim os projetos que esta envolvido

In [ ]:
cursor.execute("""
    SELECT f.nome, d.nome_departamento, p.nome_projeto
    FROM funcionarios f
    LEFT JOIN funcionarios_projetos fp ON f.id = fp.funcionario_id
    LEFT JOIN projetos p ON fp.projeto_id = p.id
    LEFT JOIN departamentos d ON d.id = f.departamento_id;
""")
cursor.fetchall()

[('Leonardo da Vinci', 'Marketing', 'Dominos'),
 ('Leonardo da Vinci', 'Marketing', 'Madero'),
 ('Leticia Almeida', 'Negócios', 'BurguerKing'),
 ('Marcos da Praia', 'RH', 'Madero'),
 ('Mohammad', 'Marketing', None),
 ('Vitor de Freitas', 'RH', None)]

In [45]:
cursor.close()
conn.close() # Encerrando a conexão com o banco de dados

## O que é DCL e Diferenças entre Categorias SQL

#### DDL (Data Definition Language):
Comandos para definir ou alterar a estrutura do banco de dados (ex: CREATE, ALTER, DROP).

#### DML (Data Manipulation Language):
Comandos para manipular os dados salvos nas tabelas (ex: INSERT, UPDATE, DELETE).

#### DQL (Data Query Language):
Comando voltado para a realização de consultas aos dados armazenados (ex: SELECT).

#### DCL (Data Control Language):
Comandos utilizados pelo Administrador do Banco de Dados (DBA) para controlar permissões, acesso e segurança do sistema.

Principais comandos: GRANT (concede privilégios) e REVOKE (revoga privilégios).